# Ulba River Flood Forecasting 



In [4]:

%pip install pandas numpy matplotlib scikit-learn torch ipykernel


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
# Step 1. Find or clone the GitHub repository locally
from pathlib import Path
import subprocess
import sys
import shutil

import os
os.environ["PATH"] = r"C:\Program Files\Git\cmd;" + os.environ.get("PATH", "")

workspace = Path.cwd()
repo = workspace if (workspace / "src" / "preprocessing.py").is_file() else workspace / "ulba-flood-forecasting"

if not (repo / "src" / "preprocessing.py").is_file():
    if shutil.which("git") is None:
        raise RuntimeError("Git не найден. Установите Git for Windows или скопируйте репозиторий ulba-flood-forecasting в эту папку.")
    subprocess.run(
        ["git", "clone", "https://github.com/lysa106/ulba-flood-forecasting.git", str(repo)],
        check=True,
    )

sys.path.insert(0, str(repo.resolve()))
from src.preprocessing import load_data, check_data, prepare_data
print("Preprocessing module imported successfully from:", repo.resolve())


Preprocessing module imported successfully from: C:\Users\Zhalg\Downloads\ulba-flood-forecasting


In [6]:
# Step 2. The five-source CSV must be stored locally (no Colab upload required)
from pathlib import Path
print("Рабочая папка:", Path.cwd())
print("Поместите CSV с пятью источниками рядом с ноутбуком или в подпапку data/.")


Рабочая папка: c:\Users\Zhalg\Downloads
Поместите CSV с пятью источниками рядом с ноутбуком или в подпапку data/.


In [7]:
# Step 3. Load the five-source CSV from the local folder
from pathlib import Path
import pandas as pd

REQUIRED_COLUMNS = [
    "date", "discharge_m3_s", "precipitation_mm", "snow_cover_mean",
    "temperature_c", "soil_moisture"
]

# Search the notebook folder, data/ and the local repository.
search_dirs = [Path.cwd(), Path.cwd() / "data", repo, repo / "data"]
csv_files = sorted({f.resolve() for d in search_dirs if d.exists() for f in d.glob("*.csv")})
valid_files = []
for file_path in csv_files:
    columns = pd.read_csv(file_path, nrows=0).columns
    if set(REQUIRED_COLUMNS).issubset(columns):
        valid_files.append(file_path)

if not valid_files:
    raise FileNotFoundError(
        "Не найден CSV с пятью источниками. Положите Ulba_5sources_PROVISIONAL_2013_2023...csv "
        "в папку с ноутбуком или в data/. Доступные файлы: " + str([str(p) for p in csv_files])
    )

preferred_files = [p for p in valid_files if "5source" in p.name.lower()]
dataset_path = (preferred_files or valid_files)[-1]
print("Using five-source file:", dataset_path)
df = load_data(dataset_path)
check_data(df)
print("\nSelected predictors:", REQUIRED_COLUMNS[1:])


Using five-source file: C:\Users\Zhalg\Downloads\Ulba_5sources_PROVISIONAL_2013_2023 (2).csv
Dataset shape: (4017, 19)
Columns: ['date', 'data_split', 'station_code', 'discharge_m3_s', 'precipitation_mm', 'imerg_halfhourly_observations', 'snow_cover_mean', 'snow_valid_pixel_count', 'snow_image_count', 'snow_missing', 'temperature_c', 'soil_moisture', 'soil_valid_cells', 'soil_quality_cells', 'soil_status_raw', 'soil_missing', 'all_five_observed', 'gridded_spatial_coverage', 'soil_spatial_coverage']

First five rows:
         date data_split  station_code  discharge_m3_s  precipitation_mm  \
0  2013-01-01      train         11164            22.7          0.000000   
1  2013-01-02      train         11164            23.1          0.063317   
2  2013-01-03      train         11164            23.5          0.000000   
3  2013-01-04      train         11164            23.4          0.031387   
4  2013-01-05      train         11164            23.3          0.051083   

   imerg_halfhourly_o

In [8]:

# Step 5: Prepare and split the dataset

df = prepare_data(df, date_column="date")

# Chronological train / validation / test split
train = df[
    (df["date"] >= "2013-01-01") &
    (df["date"] <= "2019-12-31")
].copy()

val = df[
    (df["date"] >= "2020-01-01") &
    (df["date"] <= "2021-12-31")
].copy()

test = df[
    (df["date"] >= "2022-01-01") &
    (df["date"] <= "2023-12-31")
].copy()

print("\nDATASET SPLIT")
print("Training:", len(train), "days")
print("Validation:", len(val), "days")
print("Testing:", len(test), "days")

print("\nMissing values in training:")
print(train.isna().sum())

print("\nMissing values in validation:")
print(val.isna().sum())

print("\nMissing values in testing:")
print(test.isna().sum())



Start date: 2013-01-01 00:00:00
End date: 2023-12-31 00:00:00
Total observations: 4017

DATASET SPLIT
Training: 2556 days
Validation: 731 days
Testing: 730 days

Missing values in training:
date                                0
data_split                          0
station_code                        0
discharge_m3_s                      0
precipitation_mm                    0
imerg_halfhourly_observations       0
snow_cover_mean                   413
snow_valid_pixel_count              0
snow_image_count                    0
snow_missing                        0
temperature_c                       0
soil_moisture                    1178
soil_valid_cells                    0
soil_quality_cells                  0
soil_status_raw                     0
soil_missing                        0
all_five_observed                   0
gridded_spatial_coverage            0
soil_spatial_coverage               0
dtype: int64

Missing values in validation:
date                               0
data_s

In [9]:

import numpy as np
from sklearn.preprocessing import StandardScaler

# LSTM A: gauge discharge only
FEATURES = ["discharge_m3_s"]
TARGET = "discharge_m3_s"
LOOKBACK = 14

# Fit scalers on training data only
x_scaler = StandardScaler()
y_scaler = StandardScaler()

x_scaler.fit(train[FEATURES])
y_scaler.fit(train[[TARGET]])

# Transform each period
train_x = x_scaler.transform(train[FEATURES])
val_x = x_scaler.transform(val[FEATURES])
test_x = x_scaler.transform(test[FEATURES])

train_y = y_scaler.transform(train[[TARGET]])
val_y = y_scaler.transform(val[[TARGET]])
test_y = y_scaler.transform(test[[TARGET]])

# Create 14-day input sequences
def create_sequences(X, y, lookback=14):
    X_seq, y_seq = [], []

    for i in range(lookback, len(X)):
        X_seq.append(X[i-lookback:i])
        y_seq.append(y[i])

    return np.array(X_seq), np.array(y_seq)

X_train, y_train = create_sequences(train_x, train_y, LOOKBACK)
X_val, y_val = create_sequences(val_x, val_y, LOOKBACK)
X_test, y_test = create_sequences(test_x, test_y, LOOKBACK)

print("LSTM A sequences:")
print("Training:", X_train.shape, y_train.shape)
print("Validation:", X_val.shape, y_val.shape)
print("Testing:", X_test.shape, y_test.shape)


LSTM A sequences:
Training: (2542, 14, 1) (2542, 1)
Validation: (717, 14, 1) (717, 1)
Testing: (716, 14, 1) (716, 1)


In [10]:

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
import numpy as np
import copy
import random

# Reproducibility
SEED = 17
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# Model configuration
HIDDEN_SIZE = 32
BATCH_SIZE = 32
LEARNING_RATE = 0.001
EPOCHS = 100
PATIENCE = 10

# Convert data into PyTorch tensors
X_train_t = torch.tensor(X_train, dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.float32)

X_val_t = torch.tensor(X_val, dtype=torch.float32)
y_val_t = torch.tensor(y_val, dtype=torch.float32)

train_loader = DataLoader(
    TensorDataset(X_train_t, y_train_t),
    batch_size=BATCH_SIZE,
    shuffle=True
)

# Define LSTM model
class LSTMModel(nn.Module):
    def __init__(self, input_size, hidden_size=32):
        super().__init__()

        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            batch_first=True
        )

        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        output, _ = self.lstm(x)
        last_output = output[:, -1, :]
        return self.fc(last_output)

model_a = LSTMModel(
    input_size=1,
    hidden_size=HIDDEN_SIZE
)

criterion = nn.MSELoss()
optimizer = torch.optim.Adam(
    model_a.parameters(),
    lr=LEARNING_RATE
)

# Training with early stopping
best_val_loss = float("inf")
best_weights = None
patience_counter = 0

for epoch in range(EPOCHS):
    model_a.train()
    train_losses = []

    for xb, yb in train_loader:
        optimizer.zero_grad()

        predictions = model_a(xb)
        loss = criterion(predictions, yb)

        loss.backward()
        optimizer.step()

        train_losses.append(loss.item())

    model_a.eval()

    with torch.no_grad():
        val_predictions = model_a(X_val_t)
        val_loss = criterion(
            val_predictions,
            y_val_t
        ).item()

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_weights = copy.deepcopy(model_a.state_dict())
        patience_counter = 0
    else:
        patience_counter += 1

    if (epoch + 1) % 10 == 0:
        print(
            f"Epoch {epoch+1}: "
            f"Train Loss = {np.mean(train_losses):.6f}, "
            f"Val Loss = {val_loss:.6f}"
        )

    if patience_counter >= PATIENCE:
        print(f"Early stopping at epoch {epoch+1}")
        break

# Restore best validation model
model_a.load_state_dict(best_weights)

print("\nLSTM A training completed!")


Epoch 10: Train Loss = 0.059987, Val Loss = 0.021148
Epoch 20: Train Loss = 0.051610, Val Loss = 0.018066
Epoch 30: Train Loss = 0.046784, Val Loss = 0.018983
Early stopping at epoch 32

LSTM A training completed!


In [11]:

import numpy as np
import pandas as pd
from sklearn.metrics import mean_squared_error, mean_absolute_error

# Predict on test data
model_a.eval()

with torch.no_grad():
    predictions_scaled = model_a(
        torch.tensor(X_test, dtype=torch.float32)
    ).numpy()

# Convert back to original discharge units (m3/s)
y_pred = y_scaler.inverse_transform(
    predictions_scaled
).flatten()

y_true = y_scaler.inverse_transform(
    y_test
).flatten()

# Evaluation metrics
def calculate_metrics(y_true, y_pred):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)

    denominator = np.sum((y_true - np.mean(y_true)) ** 2)
    nse = 1 - np.sum((y_true - y_pred) ** 2) / denominator

    r = np.corrcoef(y_true, y_pred)[0, 1]
    alpha = np.std(y_pred) / np.std(y_true)
    beta = np.mean(y_pred) / np.mean(y_true)

    kge = 1 - np.sqrt(
        (r - 1) ** 2 +
        (alpha - 1) ** 2 +
        (beta - 1) ** 2
    )

    # Positive PBIAS means underprediction
    pbias = 100 * np.sum(y_true - y_pred) / np.sum(y_true)

    return {
        "RMSE": rmse,
        "MAE": mae,
        "NSE": nse,
        "KGE": kge,
        "PBIAS (%)": pbias
    }

metrics_a = calculate_metrics(y_true, y_pred)

print("LSTM A — TEST RESULTS (2022–2023)")
print("-" * 40)

for metric, value in metrics_a.items():
    print(f"{metric}: {value:.4f}")

# Save test predictions with their target dates
prediction_dates = test["date"].iloc[LOOKBACK:].reset_index(drop=True)

results_a = pd.DataFrame({
    "date": prediction_dates,
    "observed_discharge": y_true,
    "predicted_discharge": y_pred
})

print("\nFirst five predictions:")
print(results_a.head())


LSTM A — TEST RESULTS (2022–2023)
----------------------------------------
RMSE: 29.2457
MAE: 10.4304
NSE: 0.9419
KGE: 0.9551
PBIAS (%): 0.4067

First five predictions:
        date  observed_discharge  predicted_discharge
0 2022-01-15                15.3            15.868942
1 2022-01-16                15.3            15.868942
2 2022-01-17                15.3            15.868942
3 2022-01-18                15.3            15.868942
4 2022-01-19                15.3            15.868942


In [12]:

# Step 9: Persistence Baseline

# Use the last observed discharge from each 14-day window
persistence_pred = x_scaler.inverse_transform(
    X_test[:, -1, :]
).flatten()

# Evaluate on exactly the same dates as LSTM A
metrics_persistence = calculate_metrics(
    y_true,
    persistence_pred
)

# Compare models
comparison = pd.DataFrame({
    "LSTM A": metrics_a,
    "Persistence": metrics_persistence
}).T

print("MODEL COMPARISON")
print(comparison.round(4))

# RMSE improvement
improvement = (
    (metrics_persistence["RMSE"] - metrics_a["RMSE"])
    / metrics_persistence["RMSE"]
) * 100

print(f"\nLSTM A RMSE improvement: {improvement:.2f}%")



MODEL COMPARISON
                RMSE      MAE     NSE     KGE  PBIAS (%)
LSTM A       29.2457  10.4304  0.9419  0.9551     0.4067
Persistence  34.3700  11.9494  0.9198  0.9599     0.0014

LSTM A RMSE improvement: 14.91%


In [13]:

# Step 10: Seasonal Climatology Baseline

import pandas as pd
import numpy as np

# Day of year represented as month-day
train_clim = train.copy()
train_clim["month_day"] = train_clim["date"].dt.strftime("%m-%d")

# Compute climatology using training data only
daily_climatology = train_clim.groupby("month_day")[
    "discharge_m3_s"
].mean()

# Match the same 716 target dates used by LSTM A
test_dates = test["date"].iloc[LOOKBACK:].reset_index(drop=True)
month_days = test_dates.dt.strftime("%m-%d")

climatology_pred = month_days.map(daily_climatology)

# Fallback for any missing calendar days
climatology_pred = climatology_pred.fillna(
    train["discharge_m3_s"].mean()
).to_numpy()

# Evaluate climatology
metrics_climatology = calculate_metrics(
    y_true,
    climatology_pred
)

# Compare all three models
comparison = pd.DataFrame({
    "LSTM A": metrics_a,
    "Persistence": metrics_persistence,
    "Seasonal Climatology": metrics_climatology
}).T

print("MODEL COMPARISON: LSTM A vs BASELINES")
print(comparison.round(4))

# Identify strongest baseline based on RMSE
baseline_rmse = {
    "Persistence": metrics_persistence["RMSE"],
    "Seasonal Climatology": metrics_climatology["RMSE"]
}

best_baseline = min(baseline_rmse, key=baseline_rmse.get)

print("\nBest baseline:", best_baseline)
print("Baseline RMSE:", round(baseline_rmse[best_baseline], 4))


MODEL COMPARISON: LSTM A vs BASELINES
                         RMSE      MAE     NSE     KGE  PBIAS (%)
LSTM A                29.2457  10.4304  0.9419  0.9551     0.4067
Persistence           34.3700  11.9494  0.9198  0.9599     0.0014
Seasonal Climatology  80.2486  48.0755  0.5629  0.6286   -28.1763

Best baseline: Persistence
Baseline RMSE: 34.37


In [14]:
# Step 11: Prepare all five data sources for LSTM B and LSTM C
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler

# LSTM B: satellite/reanalysis inputs (no gauge measurements)
GRID_FEATURES = [
    "precipitation_mm",  # GPM IMERG
    "snow_cover_mean",   # MODIS
    "temperature_c",     # ERA5-Land
    "soil_moisture"      # ESA CCI
]

# LSTM C: the same four inputs plus observed gauge discharge
COMBINED_FEATURES = [
    "discharge_m3_s",  # Kazhydromet
    *GRID_FEATURES
]

assert len(GRID_FEATURES) == 4
assert len(COMBINED_FEATURES) == 5
assert set(COMBINED_FEATURES).issubset(df.columns)

# Work chronologically. A maximum of two missing days is filled using
# PAST data only; longer gaps are excluded from sequence generation.
data_bc = df.copy().sort_values("date").reset_index(drop=True)
for feature in ["snow_cover_mean", "soil_moisture"]:
    data_bc[feature] = data_bc[feature].ffill(limit=2)

print("LSTM B inputs (4 sources):", GRID_FEATURES)
print("LSTM C inputs (5 sources):", COMBINED_FEATURES)
print("Remaining missing values:\n", data_bc[COMBINED_FEATURES].isna().sum())

# Both models must use the same 14-day windows / forecast dates.
def create_valid_sequences(data, features, lookback=14):
    X, y, dates = [], [], []

    for i in range(lookback, len(data)):
        window = data.iloc[i-lookback:i][features]
        target = data.iloc[i]["discharge_m3_s"]

        if window.isna().any().any() or pd.isna(target):
            continue

        X.append(window.to_numpy(dtype=float))
        y.append(target)
        dates.append(data.iloc[i]["date"])

    if not X:
        raise ValueError("No valid 14-day windows; verify the source CSV and gaps.")

    return np.array(X), np.array(y), pd.DatetimeIndex(dates)

X_b_raw, y_b_raw, dates_b = create_valid_sequences(
    data_bc, GRID_FEATURES, lookback=LOOKBACK
)
X_c_raw, y_c_raw, dates_c = create_valid_sequences(
    data_bc, COMBINED_FEATURES, lookback=LOOKBACK
)

assert np.array_equal(dates_b, dates_c), "LSTM B/C target dates differ"
assert np.array_equal(y_b_raw, y_c_raw), "LSTM B/C targets differ"

print("LSTM B sequences:", X_b_raw.shape)
print("LSTM C sequences:", X_c_raw.shape)
print("Total common target dates:", len(dates_c))

for name, start, end in [
    ("Train", "2013-01-01", "2019-12-31"),
    ("Validation", "2020-01-01", "2021-12-31"),
    ("Test", "2022-01-01", "2023-12-31")
]:
    count = ((dates_c >= start) & (dates_c <= end)).sum()
    print(f"{name}: {count} valid targets")

LSTM B inputs (4 sources): ['precipitation_mm', 'snow_cover_mean', 'temperature_c', 'soil_moisture']
LSTM C inputs (5 sources): ['discharge_m3_s', 'precipitation_mm', 'snow_cover_mean', 'temperature_c', 'soil_moisture']
Remaining missing values:
 discharge_m3_s         0
precipitation_mm       0
snow_cover_mean       70
temperature_c          0
soil_moisture       1704
dtype: int64
LSTM B sequences: (1890, 14, 4)
LSTM C sequences: (1890, 14, 5)
Total common target dates: 1890
Train: 1196 valid targets
Validation: 351 valid targets
Test: 343 valid targets


In [15]:

# Step 12: Scale and split LSTM B and C data

import numpy as np
from sklearn.preprocessing import StandardScaler

def prepare_model_data(X_raw, y_raw, dates):
    # Split by target date
    train_mask = dates < "2020-01-01"
    val_mask = (dates >= "2020-01-01") & (dates < "2022-01-01")
    test_mask = dates >= "2022-01-01"

    # Fit scalers on training data only
    n_features = X_raw.shape[2]

    scaler_x = StandardScaler()
    scaler_y = StandardScaler()

    X_train_raw = X_raw[train_mask]
    y_train_raw = y_raw[train_mask]

    scaler_x.fit(
        X_train_raw.reshape(-1, n_features)
    )
    scaler_y.fit(
        y_train_raw.reshape(-1, 1)
    )

    # Transform sequences
    def transform_X(X):
        original_shape = X.shape
        return scaler_x.transform(
            X.reshape(-1, n_features)
        ).reshape(original_shape).astype(np.float32)

    def transform_y(y):
        return scaler_y.transform(
            y.reshape(-1, 1)
        ).astype(np.float32)

    return {
        "X_train": transform_X(X_raw[train_mask]),
        "y_train": transform_y(y_raw[train_mask]),
        "X_val": transform_X(X_raw[val_mask]),
        "y_val": transform_y(y_raw[val_mask]),
        "X_test": transform_X(X_raw[test_mask]),
        "y_test": transform_y(y_raw[test_mask]),
        "test_dates": dates[test_mask],
        "scaler_x": scaler_x,
        "scaler_y": scaler_y
    }


data_b = prepare_model_data(
    X_b_raw, y_b_raw, dates_b
)

data_c = prepare_model_data(
    X_c_raw, y_c_raw, dates_c
)

print("LSTM B:")
print("Train:", data_b["X_train"].shape)
print("Validation:", data_b["X_val"].shape)
print("Test:", data_b["X_test"].shape)

print("\nLSTM C:")
print("Train:", data_c["X_train"].shape)
print("Validation:", data_c["X_val"].shape)
print("Test:", data_c["X_test"].shape)

print("\nSame test dates:",
      np.array_equal(
          data_b["test_dates"],
          data_c["test_dates"]
      ))


LSTM B:
Train: (1196, 14, 4)
Validation: (351, 14, 4)
Test: (343, 14, 4)

LSTM C:
Train: (1196, 14, 5)
Validation: (351, 14, 5)
Test: (343, 14, 5)

Same test dates: True


In [16]:

# Step 13: Train LSTM B and LSTM C

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
import numpy as np
import random
import copy

def train_lstm(data, model_name, seed=17):
    # Reproducibility
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    X_train_t = torch.tensor(
        data["X_train"], dtype=torch.float32
    )
    y_train_t = torch.tensor(
        data["y_train"], dtype=torch.float32
    )

    X_val_t = torch.tensor(
        data["X_val"], dtype=torch.float32
    )
    y_val_t = torch.tensor(
        data["y_val"], dtype=torch.float32
    )

    loader = DataLoader(
        TensorDataset(X_train_t, y_train_t),
        batch_size=32,
        shuffle=True
    )

    model = LSTMModel(
        input_size=data["X_train"].shape[2],
        hidden_size=32
    )

    optimizer = torch.optim.Adam(
        model.parameters(), lr=0.001
    )
    criterion = nn.MSELoss()

    best_loss = float("inf")
    best_weights = None
    patience_counter = 0

    print(f"\nTraining {model_name}...")

    for epoch in range(100):
        model.train()
        losses = []

        for xb, yb in loader:
            optimizer.zero_grad()
            predictions = model(xb)
            loss = criterion(predictions, yb)
            loss.backward()
            optimizer.step()
            losses.append(loss.item())

        model.eval()

        with torch.no_grad():
            val_loss = criterion(
                model(X_val_t), y_val_t
            ).item()

        if val_loss < best_loss:
            best_loss = val_loss
            best_weights = copy.deepcopy(
                model.state_dict()
            )
            patience_counter = 0
        else:
            patience_counter += 1

        if (epoch + 1) % 10 == 0:
            print(
                f"Epoch {epoch+1}: "
                f"Train={np.mean(losses):.5f}, "
                f"Val={val_loss:.5f}"
            )

        if patience_counter >= 10:
            print(f"Early stopping at epoch {epoch+1}")
            break

    model.load_state_dict(best_weights)
    model.eval()

    print(f"{model_name} training completed!")

    return model


# Train both models
model_b = train_lstm(data_b, "LSTM B")
model_c = train_lstm(data_c, "LSTM C")

print("\nBoth models trained successfully!")



Training LSTM B...
Epoch 10: Train=0.23859, Val=0.16752
Epoch 20: Train=0.13224, Val=0.28872
Early stopping at epoch 20
LSTM B training completed!

Training LSTM C...
Epoch 10: Train=0.05783, Val=0.02715
Epoch 20: Train=0.03685, Val=0.02061
Epoch 30: Train=0.02769, Val=0.01342
Epoch 40: Train=0.02335, Val=0.01354
Early stopping at epoch 47
LSTM C training completed!

Both models trained successfully!


In [17]:


import torch
import numpy as np

def predict_lstm(model, data):
    model.eval()

    X_test_t = torch.tensor(
        data["X_test"],
        dtype=torch.float32
    )

    with torch.no_grad():
        predictions_scaled = model(X_test_t).cpu().numpy()

    y_pred = data["scaler_y"].inverse_transform(
        predictions_scaled
    ).flatten()

    y_true = data["scaler_y"].inverse_transform(
        data["y_test"]
    ).flatten()

    return y_true, y_pred

print("predict_lstm function is ready!")


predict_lstm function is ready!


In [18]:
import numpy as np
import pandas as pd

# Predictions from LSTM B and C
y_b_true, y_b_pred = predict_lstm(model_b, data_b)
y_c_true, y_c_pred = predict_lstm(model_c, data_c)

# Build prediction table
comparison_df = pd.DataFrame({
    "date": pd.to_datetime(data_c["test_dates"]),
    "observed": y_c_true,
    "LSTM B": y_b_pred,
    "LSTM C": y_c_pred
})

# Add LSTM A predictions by date
a_df = results_a.copy()
a_df["date"] = pd.to_datetime(a_df["date"])

a_df = a_df.rename(columns={
    "predicted_discharge": "LSTM A"
})

comparison_df = comparison_df.merge(
    a_df[["date", "LSTM A"]],
    on="date",
    how="inner",
    validate="one_to_one"
)

# Persistence: previous day's observed discharge
previous_discharge = df[
    ["date", "discharge_m3_s"]
].copy()

previous_discharge["date"] = (
    previous_discharge["date"] + pd.Timedelta(days=1)
)

previous_discharge = previous_discharge.rename(
    columns={"discharge_m3_s": "Persistence"}
)

comparison_df = comparison_df.merge(
    previous_discharge,
    on="date",
    how="left",
    validate="one_to_one"
)

# Seasonal climatology based on training data only
comparison_df["month_day"] = (
    comparison_df["date"].dt.strftime("%m-%d")
)

comparison_df["Seasonal Climatology"] = (
    comparison_df["month_day"]
    .map(daily_climatology)
    .fillna(train["discharge_m3_s"].mean())
)

# Ensure all predictions are available
model_names = [
    "LSTM A",
    "LSTM B",
    "LSTM C",
    "Persistence",
    "Seasonal Climatology"
]

comparison_df = comparison_df.dropna(
    subset=["observed"] + model_names
).sort_values("date").reset_index(drop=True)

if comparison_df.empty:
    raise ValueError("No common test dates found.")

# Check that B and C use the same observations
b_observed = pd.Series(
    y_b_true,
    index=pd.to_datetime(data_b["test_dates"])
)
aligned_b = b_observed.loc[comparison_df["date"]].to_numpy()

assert np.allclose(
    comparison_df["observed"].to_numpy(),
    aligned_b
)

# Calculate metrics on identical dates
y_common = comparison_df["observed"].to_numpy()

all_metrics = {
    name: calculate_metrics(
        y_common,
        comparison_df[name].to_numpy()
    )
    for name in model_names
}

comparison_all = pd.DataFrame(all_metrics).T

print("FAIR PILOT MODEL COMPARISON")
print("Common test dates:", len(comparison_df))
print(comparison_all.round(4))

rmse_c = all_metrics["LSTM C"]["RMSE"]
rmse_p = all_metrics["Persistence"]["RMSE"]

improvement_c = 100 * (rmse_p - rmse_c) / rmse_p

print(
    f"\nLSTM C improvement over Persistence: "
    f"{improvement_c:.2f}%"
)


FAIR PILOT MODEL COMPARISON
Common test dates: 343
                          RMSE      MAE     NSE     KGE  PBIAS (%)
LSTM A                 29.2008  10.5509  0.9344  0.9561    -1.1457
LSTM B                100.0009  65.2262  0.2303  0.4085   -22.0233
LSTM C                 22.3964  10.7774  0.9614  0.9326     2.0787
Persistence            31.9819  10.9102  0.9213  0.9581    -1.2609
Seasonal Climatology   75.7178  51.8309  0.5587  0.5433   -28.4171

LSTM C improvement over Persistence: 29.97%


In [19]:
# Step 15: Five-source ablation study on IDENTICAL train/val/test dates
import numpy as np
import pandas as pd

SEEDS = [17, 29, 41]

ABLATION_CONFIGS = {
    "All five sources": COMBINED_FEATURES,
    "Without precipitation": [x for x in COMBINED_FEATURES if x != "precipitation_mm"],
    "Without snow cover": [x for x in COMBINED_FEATURES if x != "snow_cover_mean"],
    "Without temperature": [x for x in COMBINED_FEATURES if x != "temperature_c"],
    "Without soil moisture": [x for x in COMBINED_FEATURES if x != "soil_moisture"],
}

# Align the entire train/validation/test dataset, not just the final test set.
# Otherwise a reduced-feature model may benefit from extra training windows.
common_sequence_dates = pd.DatetimeIndex(dates_c)
common_ablation_dates = pd.DatetimeIndex(comparison_df["date"])
ablation_results = []

for config_name, features in ABLATION_CONFIGS.items():
    print(f"\nConfiguration: {config_name}; inputs={len(features)}")

    X_raw, y_raw, dates = create_valid_sequences(
        data_bc, features, lookback=LOOKBACK
    )
    idx = dates.get_indexer(common_sequence_dates)
    if (idx < 0).any():
        raise ValueError(f"Missing common sequence dates for {config_name}")

    prepared = prepare_model_data(
        X_raw[idx], y_raw[idx], dates[idx]
    )
    assert np.array_equal(prepared["test_dates"], data_c["test_dates"])

    run_rmse = []
    for seed in SEEDS:
        model = train_lstm(
            prepared, f"{config_name} (seed={seed})", seed=seed
        )
        y_actual, y_predicted = predict_lstm(model, prepared)
        pred_series = pd.Series(y_predicted, index=prepared["test_dates"])
        actual_series = pd.Series(y_actual, index=prepared["test_dates"])
        aligned_pred = pred_series.reindex(common_ablation_dates)
        aligned_actual = actual_series.reindex(common_ablation_dates)

        if aligned_pred.isna().any() or aligned_actual.isna().any():
            raise ValueError(f"Missing matched test predictions for {config_name}")
        rmse = calculate_metrics(
            aligned_actual.to_numpy(), aligned_pred.to_numpy()
        )["RMSE"]
        run_rmse.append(rmse)
        print(f"Seed {seed}: RMSE = {rmse:.4f}")

    ablation_results.append({
        "Configuration": config_name,
        "RMSE mean": np.mean(run_rmse),
        "RMSE SD": np.std(run_rmse, ddof=1)
    })

ablation_table = pd.DataFrame(ablation_results)
baseline_rmse = ablation_table.loc[
    ablation_table["Configuration"] == "All five sources", "RMSE mean"
].iloc[0]
ablation_table["RMSE change"] = ablation_table["RMSE mean"] - baseline_rmse

print("\nFIVE-SOURCE ABLATION STUDY RESULTS")
print(ablation_table.round(4).to_string(index=False))


Configuration: All five sources; inputs=5

Training All five sources (seed=17)...
Epoch 10: Train=0.05783, Val=0.02715
Epoch 20: Train=0.03685, Val=0.02061
Epoch 30: Train=0.02769, Val=0.01342
Epoch 40: Train=0.02335, Val=0.01354
Early stopping at epoch 47
All five sources (seed=17) training completed!
Seed 17: RMSE = 22.3964

Training All five sources (seed=29)...
Epoch 10: Train=0.05357, Val=0.02561
Epoch 20: Train=0.03559, Val=0.01614
Epoch 30: Train=0.02733, Val=0.01787
Early stopping at epoch 33
All five sources (seed=29) training completed!
Seed 29: RMSE = 24.0073

Training All five sources (seed=41)...
Epoch 10: Train=0.05285, Val=0.02434
Epoch 20: Train=0.03620, Val=0.01756
Epoch 30: Train=0.02699, Val=0.01723
Epoch 40: Train=0.02165, Val=0.01931
Epoch 50: Train=0.01558, Val=0.01852
Early stopping at epoch 57
All five sources (seed=41) training completed!
Seed 41: RMSE = 24.9228

Configuration: Without precipitation; inputs=4

Training Without precipitation (seed=17)...
Epoch 

In [20]:

# Step 16: Statistical Testing
# Moving Block Bootstrap + Paired Block Sign-Flip Test

import numpy as np
import pandas as pd

N_BOOTSTRAP = 1000
BLOCK_SIZE = 14
SEED = 42

def rmse(y, pred):
    return np.sqrt(np.mean((y - pred) ** 2))


def rmse_reduction(y, pred_reference, pred_model):
    reference_rmse = rmse(y, pred_reference)
    model_rmse = rmse(y, pred_model)

    return 100 * (
        reference_rmse - model_rmse
    ) / reference_rmse


def statistical_comparison(
    y, reference, model, n_bootstrap=1000,
    block_size=14, seed=42
):
    rng = np.random.default_rng(seed)
    n = len(y)

    observed_reduction = rmse_reduction(
        y, reference, model
    )

    # Moving block bootstrap
    bootstrap_reductions = []

    for _ in range(n_bootstrap):
        indices = []

        while len(indices) < n:
            start = rng.integers(
                0, max(1, n - block_size + 1)
            )
            indices.extend(
                range(start, min(start + block_size, n))
            )

        indices = np.array(indices[:n])

        reduction = rmse_reduction(
            y[indices],
            reference[indices],
            model[indices]
        )

        bootstrap_reductions.append(reduction)

    ci_low, ci_high = np.percentile(
        bootstrap_reductions, [2.5, 97.5]
    )

    # Paired block sign-flip test on squared error differences
    loss_difference = (
        (y - reference) ** 2
        - (y - model) ** 2
    )

    observed_stat = np.abs(
        np.mean(loss_difference)
    )

    blocks = [
        loss_difference[i:i + block_size]
        for i in range(0, n, block_size)
    ]

    extreme_count = 0

    for _ in range(n_bootstrap):
        flipped = np.concatenate([
            block * rng.choice([-1, 1])
            for block in blocks
        ])

        test_stat = np.abs(np.mean(flipped))

        if test_stat >= observed_stat:
            extreme_count += 1

    p_value = (
        extreme_count + 1
    ) / (n_bootstrap + 1)

    return {
        "RMSE reduction (%)": observed_reduction,
        "CI lower (%)": ci_low,
        "CI upper (%)": ci_high,
        "p-value": p_value
    }


# Use the matched comparison dataset
y = comparison_df["observed"].to_numpy()

comparisons = {
    "C vs Persistence": "Persistence",
    "C vs LSTM A": "LSTM A",
    "C vs LSTM B": "LSTM B"
}

stat_results = []

for label, reference_name in comparisons.items():
    result = statistical_comparison(
        y=y,
        reference=comparison_df[
            reference_name
        ].to_numpy(),
        model=comparison_df[
            "LSTM C"
        ].to_numpy(),
        n_bootstrap=N_BOOTSTRAP,
        block_size=BLOCK_SIZE,
        seed=SEED
    )

    result["Comparison"] = label
    stat_results.append(result)

stats_table = pd.DataFrame(stat_results)

print("STATISTICAL TEST RESULTS")
print(stats_table.round(4).to_string(index=False))


STATISTICAL TEST RESULTS
 RMSE reduction (%)  CI lower (%)  CI upper (%)  p-value       Comparison
            29.9718      -16.7980       41.5633   0.0979 C vs Persistence
            23.3022      -17.2881       29.3931   0.0929      C vs LSTM A
            77.6038       74.5296       87.3172   0.0010      C vs LSTM B


In [21]:
# Step 17: Save five-source results in a separate folder (do not overwrite pilot)
from pathlib import Path
import json
import pandas as pd

result_dir = Path("results_5sources")
result_dir.mkdir(parents=True, exist_ok=True)

comparison_all.to_csv(result_dir / "model_comparison.csv", index_label="Model")
comparison_df.to_csv(result_dir / "test_predictions.csv", index=False)
ablation_table.to_csv(result_dir / "ablation_results.csv", index=False)
stats_table.to_csv(result_dir / "statistical_tests.csv", index=False)

run_config = {
    "experiment": "five_source_provisional",
    "dataset": dataset_path.name,
    "five_source_predictors": COMBINED_FEATURES,
    "gridded_predictors": GRID_FEATURES,
    "gauge_predictors": FEATURES,
    "lookback_days": LOOKBACK,
    "forecast_horizon_days": 1,
    "forward_fill_limit_days": 2,
    "training_period": "2013-01-01/2019-12-31",
    "validation_period": "2020-01-01/2021-12-31",
    "testing_period": "2022-01-01/2023-12-31",
    "matched_test_dates": len(comparison_df),
    "ablation_seeds": SEEDS,
    "caveat": "Provisional spatial coverage, not a verified upstream catchment."
}
(result_dir / "run_config.json").write_text(
    json.dumps(run_config, ensure_ascii=False, indent=2), encoding="utf-8"
)

print("Five-source results saved:", [p.name for p in result_dir.iterdir()])

Five-source results saved: ['ablation_results.csv', 'model_comparison.csv', 'run_config.json', 'statistical_tests.csv', 'test_predictions.csv']


In [22]:
# Step 18. Create a ZIP archive locally (no Colab download required)
import shutil
from pathlib import Path

archive_path = shutil.make_archive("ulba_5sources_results", "zip", "results_5sources")
print("Результаты сохранены:", Path(archive_path).resolve())
print("Папка с отдельными файлами:", Path("results_5sources").resolve())


Результаты сохранены: C:\Users\Zhalg\Downloads\ulba_5sources_results.zip
Папка с отдельными файлами: C:\Users\Zhalg\Downloads\results_5sources


In [23]:

# Final check of all experiments

required = [
    "model_a",
    "model_b",
    "model_c",
    "comparison_all",
    "ablation_table",
    "stats_table",
    "comparison_df"
]

missing = [x for x in required if x not in globals()]

if missing:
    print("Missing:", missing)
else:
    print("ALL EXPERIMENTS COMPLETED SUCCESSFULLY!")
    print("\nModel comparison:")
    print(comparison_all.round(4))

    print("\nAblation study:")
    print(ablation_table.round(4))

    print("\nStatistical tests:")
    print(stats_table.round(4))


ALL EXPERIMENTS COMPLETED SUCCESSFULLY!

Model comparison:
                          RMSE      MAE     NSE     KGE  PBIAS (%)
LSTM A                 29.2008  10.5509  0.9344  0.9561    -1.1457
LSTM B                100.0009  65.2262  0.2303  0.4085   -22.0233
LSTM C                 22.3964  10.7774  0.9614  0.9326     2.0787
Persistence            31.9819  10.9102  0.9213  0.9581    -1.2609
Seasonal Climatology   75.7178  51.8309  0.5587  0.5433   -28.4171

Ablation study:
           Configuration  RMSE mean  RMSE SD  RMSE change
0       All five sources    23.7755   1.2791       0.0000
1  Without precipitation    31.5657   1.5005       7.7902
2     Without snow cover    21.9752   0.7889      -1.8003
3    Without temperature    24.2920   0.8948       0.5165
4  Without soil moisture    22.8583   0.7734      -0.9172

Statistical tests:
   RMSE reduction (%)  CI lower (%)  CI upper (%)  p-value        Comparison
0             29.9718      -16.7980       41.5633   0.0979  C vs Persistence
